# Kimi-Linear (GDN-2) — Pretrain + SFT on Google Colab

This notebook runs the **entire** training pipeline of this project on **Google
Colab**, end to end, in two phases:

1. **Phase 1 — Pretrain**: plain next-token language modeling on a code corpus
   (every token supervised), to teach the model general Python.
2. **Phase 2 — SFT**: completion-masked supervised fine-tuning on **MBPP**
   (`task → code`), continuing from the pretrained weights.
3. **Evaluate**: held-out perplexity **and** functional `pass@k` — we *execute*
   the generated programs against MBPP's unit tests.

It imports the real `codegen/` package (nothing is re-implemented here), so it can
never drift from the source. The single `train(cfg)` call performs **both phases
automatically** because we set a pretrain corpus on the config.

> **▶ Before you run:** set a GPU runtime — **Runtime → Change runtime type →
> Hardware accelerator → GPU**. The pipeline also runs on CPU (much slower); the
> setup cells detect which you have.

## 0. Colab setup — clone the repo and install dependencies

The first cell clones this project (so `codegen/`, the model files, etc. are present)
and `cd`s into it. The second installs the libraries plus a **CUDA build of JAX**
for the Colab GPU. Both are no-ops when you run the notebook locally from inside the
repo.

In [ ]:
import os, sys, subprocess, shutil

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/wisnunugroho21/nugie-coding-llm-agent.git"
BRANCH   = "dev"

if IN_COLAB and not os.path.isdir("nugie-coding-llm-agent"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL], check=True)
if IN_COLAB:
    os.chdir("nugie-coding-llm-agent")

sys.path.insert(0, os.path.abspath("."))   # make codegen/ + model files importable

# Detect an NVIDIA GPU so we install the right JAX build.
HAS_GPU = shutil.which("nvidia-smi") is not None and \
          subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("in Colab:", IN_COLAB, "| GPU:", HAS_GPU, "| cwd:", os.getcwd())

In [ ]:
# Install dependencies. On Colab this (re)installs into the session.
if IN_COLAB:
    base = ["flax>=0.12.0", "optax>=0.2.8", "orbax-checkpoint>=0.11.0",
            "numpy>=2.0", "datasets>=3.0", "tokenizers>=0.22", "tqdm>=4.67"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *base], check=True)
    # JAX: CUDA build for a GPU runtime, plain CPU build otherwise.
    jax_spec = "jax[cuda12]" if HAS_GPU else "jax"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", jax_spec], check=True)
print("dependencies ready")

In [ ]:
# Environment + imports. Allow the sandbox to execute generated code for pass@k
# (it is *our own* code, run against MBPP's asserts). Force CPU only if no GPU.
if not HAS_GPU:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ["CODEGEN_ALLOW_EXEC"] = "1"

import warnings; warnings.filterwarnings("ignore")
import jax
import flax.nnx as nnx
print("jax", jax.__version__, "| devices:", jax.devices())

## 1. Configuration — preset, corpus, and step budgets

A single `TrainConfig` describes the whole run. Setting `pretrain_corpus` is what
turns on **Phase 1**; `train(cfg)` then does pretrain → SFT in one call.

Knobs you can tune:

* **`PRESET`** — `"small"` (~200M params, needs a GPU) or `"tiny"` (CPU plumbing).
  Defaults to `small` when a GPU is present.
* **`PRETRAIN_CORPUS`** — a HuggingFace dataset id (streamed) **or** a local directory
  of source files. Default is the `codeparrot` validation split (matches the README).
* **`PRETRAIN_STEPS` / `SFT_STEPS`** — capped so a Colab session finishes; raise them
  (or set to `None` to use the preset's epoch schedule) for a stronger model.

In [ ]:
from codegen.config import get_preset

PRESET          = "small" if HAS_GPU else "tiny"
PRETRAIN_CORPUS = "codeparrot/codeparrot-clean-valid"  # HF id, or a local dir of .py
PRETRAIN_HF_FIELD = "content"
PRETRAIN_MAX_DOCS = 2000     # cap documents streamed from the corpus (keeps it in RAM)
PRETRAIN_STEPS    = 150      # Phase-1 steps (None -> preset epoch schedule)
SFT_STEPS         = 300      # Phase-2 steps (None -> preset epoch schedule)

cfg = get_preset(PRESET)

# --- Phase 1 (pretrain) ---
cfg.pretrain_corpus    = PRETRAIN_CORPUS
cfg.pretrain_hf_field  = PRETRAIN_HF_FIELD
cfg.pretrain_max_docs  = PRETRAIN_MAX_DOCS
cfg.pretrain_max_steps = PRETRAIN_STEPS
cfg.pretrain_lr        = cfg.lr

# --- Phase 2 (SFT) ---
cfg.max_steps = SFT_STEPS

# Evaluate/checkpoint a few times within the capped budget, and keep pass@k small.
cfg.eval_every        = max(SFT_STEPS // 3, 10)
cfg.ckpt_every        = max(SFT_STEPS // 2, 10)
cfg.eval_max_problems = 20
cfg.eval_n_samples    = 5

print("preset           :", cfg.name)
print("pretrain corpus  :", cfg.pretrain_corpus, f"(<= {cfg.pretrain_max_docs} docs)")
print("pretrain steps   :", cfg.pretrain_max_steps)
print("SFT steps        :", cfg.max_steps)
print("model            :", f"d_model={cfg.model.d_model}, n_layers={cfg.model.n_layers}, "
      f"experts={cfg.model.moe_n_routed} (top-{cfg.model.moe_top_k})")

## 2. Tokenizer — byte-level BPE

We train a byte-level BPE on the MBPP corpus (no `<unk>`, indentation preserved).
Because it is byte-level it still encodes the pretrain corpus losslessly — just less
compactly. For a production run you would train the BPE on the *pretrain* corpus too;
here we keep it on MBPP for speed.

In [ ]:
from codegen.tokenizer import CodeTokenizer, train_tokenizer, _mbpp_corpus

if not os.path.exists(cfg.tokenizer_path):
    train_tokenizer(_mbpp_corpus(cfg), cfg.vocab_size, save_path=cfg.tokenizer_path)
tok = CodeTokenizer.load(cfg.tokenizer_path)
print("vocab size:", tok.vocab_size, "| pad_id:", tok.pad_id, "| eos_id:", tok.eos_id)

## 3. Peek at both data formats

The two phases use **different supervision**:

* **Pretrain** packs the corpus into fixed-length blocks with an **all-ones** loss mask
  (every token is a target — standard LM).
* **SFT** frames each MBPP task as a prompt (description + asserts) followed by the
  reference solution, and supervises **only the completion span** (prompt + padding are
  masked out).

In [ ]:
from codegen.data import load_pretrain_dataset, load_sft_datasets, load_eval_problems

# Phase-1 packed LM blocks. We peek with a SMALL temporary doc cap so this preview
# is fast — train() below streams the full corpus (PRETRAIN_MAX_DOCS) when it runs.
cfg.model.vocab_size = tok.vocab_size
_peek = cfg.pretrain_max_docs
cfg.pretrain_max_docs = min(_peek, 50)
pre_ds = load_pretrain_dataset(cfg, tok)
cfg.pretrain_max_docs = _peek  # restore the real cap for training
print(f"pretrain (preview): {len(pre_ds)} blocks x seq_len {cfg.train_seq_len}, "
      f"loss-mask mean = {pre_ds.loss_mask.mean():.2f} (every token supervised)\n")

# Phase-2 SFT example: the prompt fed to the model (loss applies only to the completion).
prob = load_eval_problems(cfg, split="test", max_problems=1)[0]
print("----- SFT prompt fed to the model -----\n")
print(prob.prompt)

## 4. Train — Phase 1 (pretrain) → Phase 2 (SFT)

`train(cfg)` runs the real loop for both phases back to back:

* **Phase 1** trains plain next-token LM on the packed corpus with its own
  warmup→cosine schedule, then saves `ckpt-pretrain`.
* **Phase 2** continues those weights as MBPP SFT (completion-masked loss + MoE aux
  loss + aux-loss-free router-bias balancing), evaluates held-out perplexity, and saves
  the best model to `ckpt-best` (plus `ckpt-last`).

Watch the `[pretrain]` then `[sft]` step logs; loss should fall in both. It returns the
best-checkpoint path.

In [ ]:
from codegen.train import train

best_ckpt = train(cfg)    # <-- runs PRETRAIN then SFT
print("\nbest checkpoint:", best_ckpt)
print("pretrain checkpoint:", os.path.join(cfg.out_dir, "ckpt-pretrain"))

## 5. Reload the best checkpoint and measure perplexity

We persist only the model **state**; `load_checkpoint` rebuilds the structure abstractly
and restores the arrays. Perplexity on the held-out MBPP completions is cheap (no code
execution).

In [ ]:
from codegen.checkpointing import load_checkpoint
from codegen.evaluate import evaluate_perplexity

model = load_checkpoint(cfg.model, best_ckpt)
_, val_ds = load_sft_datasets(cfg, tok)
ppl = evaluate_perplexity(model, val_ds, cfg.batch_size)
print(f"held-out perplexity: {ppl:.2f}")

## 6. Sample a completion

Batched temperature / top-p decoding via the model's streaming `step` (the same path
`pass@k` uses). One sampled program for the problem above:

In [ ]:
from codegen.sampling import generate_completions
from codegen.evaluate import STOP_STRINGS

completion = generate_completions(
    model, tok, prob.prompt, n_samples=1,
    stops=STOP_STRINGS, max_new_tokens=cfg.eval_max_new_tokens,
    temperature=cfg.eval_temperature, top_p=cfg.eval_top_p,
    key=jax.random.PRNGKey(0),
)[0]
print("----- model completion -----\n")
print(completion)

## 7. Functional `pass@k`

For each test task we sample `n_samples` programs, **execute** each against the task's
`assert` tests in a sandboxed subprocess (wall-clock timeout; POSIX resource limits
where available), count how many pass (`c`), and report the unbiased
`pass@k = 1 - C(n-c, k)/C(n, k)` (Chen et al., 2021), averaged over tasks.

> With a short, capped Colab run the scores will be low — that is expected. Raise
> `PRETRAIN_STEPS` / `SFT_STEPS` (and use the `small` preset on a good GPU) for real
> numbers.

In [ ]:
from codegen.evaluate import evaluate_pass_at_k

report = evaluate_pass_at_k(model, tok, cfg, verbose=False)
for k, v in report["pass_at_k"].items():
    print(f"pass@{k} = {v:.3f}")
print("exec status counts:", report["status_counts"])

## 8. Save your results off Colab

Colab disks are ephemeral — the `runs/<preset>/` directory (checkpoints, tokenizer,
logs) disappears when the session ends. Persist it before you leave.

**Option A — download a zip:**
```python
import shutil
from google.colab import files
shutil.make_archive("run_artifacts", "zip", cfg.out_dir)
files.download("run_artifacts.zip")
```

**Option B — copy to Google Drive:**
```python
from google.colab import drive; drive.mount("/content/drive")
import shutil; shutil.copytree(cfg.out_dir, "/content/drive/MyDrive/kimi_run", dirs_exist_ok=True)
```

### Scaling up
Every stage ran end to end. For a stronger model: raise `PRETRAIN_STEPS` /
`SFT_STEPS` (or set them to `None` for the preset's full epoch schedule), increase
`PRETRAIN_MAX_DOCS`, point `PRETRAIN_CORPUS` at a larger code dataset, and train the BPE
tokenizer on that same corpus. See [`README.md`](README.md) and [`usage.md`](usage.md)
for the full set of options and the equivalent command-line workflow.